<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №17


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

В данной версии к существующей иерархии добавлена явная реализация интерфейсов ITrackable и IInsurable, а также внедрен паттерн управления зависимостями (Dependency Injection).

Явная реализация интерфейсов: Для скрытия служебных методов трекинга и расчета страховых взносов от прямого вызова через базовые типы классов, методы интерфейсов ITrackable и IInsurable реализованы явно в классах ExpressDelivery и Pickup.

Управление зависимостями (Dependency Injection): Создан класс OrderProcessingService, который принимает абстракции (ShippingOption, ITrackable, IInsurable) через конструктор и управляет логикой оформления и контроля доставки без прямой жесткой связности с конкретными классами.

Расширение свойств и методов: К базовым и производным классам добавлено по 3-4 новых атрибута и метода для контроля статусов, временных рамок и условий безопасности.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

public class Order
{
    private int _orderId;
    private double _packageWeight;
    private string _receiverName;
    private decimal _declaredValue;
    private bool _isFragile;

    public int OrderId
    {
        get { return _orderId; }
        set { _orderId = value > 0 ? value : 1; }
    }

    public double PackageWeight
    {
        get { return _packageWeight; }
        set { _packageWeight = value > 0 ? value : 0.1; }
    }

    public string ReceiverName
    {
        get { return _receiverName; }
        set { _receiverName = string.IsNullOrWhiteSpace(value) ? "Unknown" : value; }
    }

    public decimal DeclaredValue
    {
        get { return _declaredValue; }
        set { _declaredValue = value >= 0 ? value : 0; }
    }

    public bool IsFragile
    {
        get { return _isFragile; }
        set { _isFragile = value; }
    }

    public Order(int orderId, double packageWeight, string receiverName, decimal declaredValue, bool isFragile = false)
    {
        OrderId = orderId;
        PackageWeight = packageWeight;
        ReceiverName = receiverName;
        DeclaredValue = declaredValue;
        IsFragile = isFragile;
    }

    public string GetOrderSummary()
    {
        return $"Order #{OrderId} | Receiver: {ReceiverName} | Weight: {PackageWeight}kg | Value: {DeclaredValue:C}";
    }

    public void UpdateWeight(double newWeight)
    {
        PackageWeight = newWeight;
    }
}

// 1. Интерфейсы для доставки
public interface ITrackable
{
    string TrackingNumber { get; set; }
    string GetTrackingStatus();
    void UpdateStatus(string status);
}

public interface IInsurable
{
    decimal InsuranceRate { get; set; }
    decimal CalculateInsuranceCost(Order order);
}

// 2. Базовый класс ShippingOption
public class ShippingOption
{
    private int _id;
    private string _name;
    private decimal _baseCost;
    private double _maxWeight;
    private string _description;
    private bool _isActive;
    private int _priorityLevel;
    private decimal _handlingFee;

    public int Id
    {
        get { return _id; }
        set { _id = value; }
    }

    public string Name
    {
        get { return _name; }
        set { _name = string.IsNullOrWhiteSpace(value) ? "Unknown" : value; }
    }

    public decimal BaseCost
    {
        get { return _baseCost; }
        set { _baseCost = value >= 0 ? value : 0; }
    }

    public double MaxWeight
    {
        get { return _maxWeight; }
        set { _maxWeight = value > 0 ? value : 1000; }
    }

    public string Description
    {
        get { return _description; }
        set { _description = string.IsNullOrWhiteSpace(value) ? "-" : value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public int PriorityLevel
    {
        get { return _priorityLevel; }
        set { _priorityLevel = value > 0 ? value : 1; }
    }

    public decimal HandlingFee
    {
        get { return _handlingFee; }
        set { _handlingFee = value >= 0 ? value : 0; }
    }

    public ShippingOption(int id, string name, decimal baseCost, double maxWeight)
    {
        Id = id;
        Name = name;
        BaseCost = baseCost;
        MaxWeight = maxWeight;
        IsActive = true;
        Description = "Standard shipping option";
        PriorityLevel = 1;
        HandlingFee = 50m;
    }

    public virtual decimal CalculateCost(Order order)
    {
        return BaseCost + (order.PackageWeight > 1.0 ? (decimal)(order.PackageWeight - 1.0) * 50m : 0m);
    }

    public virtual string GetDeliveryDetails()
    {
        return $"[{Id}] {Name} | Base Cost: {BaseCost:C} | Max Weight: {MaxWeight}kg";
    }

    public bool CheckAvailability(Order order)
    {
        return IsActive && order.PackageWeight <= MaxWeight;
    }

    public void ApplyDiscount(decimal percent)
    {
        if (percent > 0 && percent <= 100)
        {
            BaseCost -= BaseCost * (percent / 100);
        }
    }

    public void ToggleActiveStatus()
    {
        IsActive = !IsActive;
    }
}

// 3. Промежуточный класс CourierDelivery
public class CourierDelivery : ShippingOption
{
    private string _courierName;
    private string _vehicleType;
    private string _contactPhone;
    private bool _isContactlessAllowed;

    public string CourierName
    {
        get { return _courierName; }
        set { _courierName = string.IsNullOrWhiteSpace(value) ? "Pending" : value; }
    }

    public string VehicleType
    {
        get { return _vehicleType; }
        set { _vehicleType = string.IsNullOrWhiteSpace(value) ? "Van" : value; }
    }

    public string ContactPhone
    {
        get { return _contactPhone; }
        set { _contactPhone = string.IsNullOrWhiteSpace(value) ? "N/A" : value; }
    }

    public bool IsContactlessAllowed
    {
        get { return _isContactlessAllowed; }
        set { _isContactlessAllowed = value; }
    }

    public CourierDelivery(int id, string name, decimal baseCost, double maxWeight) 
        : base(id, name, baseCost, maxWeight)
    {
        CourierName = "Not assigned";
        VehicleType = "Van";
        ContactPhone = "000-000";
        IsContactlessAllowed = true;
    }

    public void AssignCourier(string name, string phone)
    {
        CourierName = name;
        ContactPhone = phone;
    }

    public string GetCourierInfo()
    {
        return $"Courier: {CourierName} | Phone: {ContactPhone} | Vehicle: {VehicleType}";
    }

    public void UpgradeVehicle(string newVehicle)
    {
        VehicleType = newVehicle;
    }
}

// 4. Класс ExpressDelivery с ЯВНОЙ реализацией интерфейсов ITrackable и IInsurable
public class ExpressDelivery : CourierDelivery, ITrackable, IInsurable
{
    private int _minTime;
    private decimal _surcharge;
    private string _trackingNumber;
    private decimal _insuranceRate;
    private string _currentStatus;

    public int MinTime
    {
        get { return _minTime; }
        set { _minTime = value > 0 ? value : 1; }
    }

    public decimal Surcharge
    {
        get { return _surcharge; }
        set { _surcharge = value >= 0 ? value : 0; }
    }

    public ExpressDelivery(int id, string name, decimal baseCost, double maxWeight, int minTime, decimal surcharge) 
        : base(id, name, baseCost, maxWeight)
    {
        MinTime = minTime;
        Surcharge = surcharge;
        _trackingNumber = "EXP-DEFAULT";
        _insuranceRate = 0.05m;
        _currentStatus = "Created";
    }

    // ЯВНАЯ РЕАЛИЗАЦИЯ ITrackable
    string ITrackable.TrackingNumber
    {
        get { return _trackingNumber; }
        set { _trackingNumber = value; }
    }

    string ITrackable.GetTrackingStatus()
    {
        return $"[EXPRESS TRACKING #{_trackingNumber}] Current Status: {_currentStatus}";
    }

    void ITrackable.UpdateStatus(string status)
    {
        _currentStatus = status;
    }

    // ЯВНАЯ РЕАЛИЗАЦИЯ IInsurable
    decimal IInsurable.InsuranceRate
    {
        get { return _insuranceRate; }
        set { _insuranceRate = value; }
    }

    decimal IInsurable.CalculateInsuranceCost(Order order)
    {
        return order.DeclaredValue * _insuranceRate;
    }

    public override decimal CalculateCost(Order order)
    {
        // Вызов явной реализации страхования внутри класса
        decimal insurance = ((IInsurable)this).CalculateInsuranceCost(order);
        return base.CalculateCost(order) + Surcharge + insurance;
    }
}

// 5. Класс Pickup с ЯВНОЙ реализацией ITrackable
public class Pickup : ShippingOption, ITrackable
{
    private string _address;
    private int _storageDays;
    private string _trackingNumber;
    private string _currentStatus;

    public string Address
    {
        get { return _address; }
        set { _address = string.IsNullOrWhiteSpace(value) ? "Main Hub" : value; }
    }

    public int StorageDays
    {
        get { return _storageDays; }
        set { _storageDays = value > 0 ? value : 3; }
    }

    public Pickup(int id, string name, decimal baseCost, double maxWeight, string address) 
        : base(id, name, baseCost, maxWeight)
    {
        Address = address;
        StorageDays = 5;
        _trackingNumber = "PKP-DEFAULT";
        _currentStatus = "Awaiting arrival";
    }

    // ЯВНАЯ РЕАЛИЗАЦИЯ ITrackable
    string ITrackable.TrackingNumber
    {
        get { return _trackingNumber; }
        set { _trackingNumber = value; }
    }

    string ITrackable.GetTrackingStatus()
    {
        return $"[PICKUP TRACKING #{_trackingNumber}] Status: {_currentStatus} at {Address}";
    }

    void ITrackable.UpdateStatus(string status)
    {
        _currentStatus = status;
    }

    public override decimal CalculateCost(Order order)
    {
        return BaseCost;
    }

    public void ExtendStorage(int days)
    {
        if (days > 0) StorageDays += days;
    }
}

// 6. УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ (Dependency Injection)
// Сервис обработки заказов принимает зависимости через конструктор
public class OrderProcessingService
{
    private readonly ShippingOption _shippingOption;

    // Внедрение зависимости через конструктор
    public OrderProcessingService(ShippingOption shippingOption)
    {
        _shippingOption = shippingOption ?? throw new ArgumentNullException(nameof(shippingOption));
    }

    public void ProcessOrder(Order order)
    {
        Console.WriteLine($"\n--- Processing Order #{order.OrderId} via Service ---");
        Console.WriteLine(order.GetOrderSummary());
        Console.WriteLine(_shippingOption.GetDeliveryDetails());

        if (_shippingOption.CheckAvailability(order))
        {
            decimal totalCost = _shippingOption.CalculateCost(order);
            Console.WriteLine($"Calculated Total Cost: {totalCost:C}");

            // Если объект поддерживает явный интерфейс ITrackable, используем его
            if (_shippingOption is ITrackable trackable)
            {
                trackable.UpdateStatus("In Transit");
                Console.WriteLine(trackable.GetTrackingStatus());
            }

            // Если объект поддерживает явный интерфейс IInsurable, рассчитываем страховку отдельно
            if (_shippingOption is IInsurable insurable)
            {
                decimal insCost = insurable.CalculateInsuranceCost(order);
                Console.WriteLine($"Insurance Fee Included: {insCost:C} (Rate: {insurable.InsuranceRate:P})");
            }
        }
        else
        {
            Console.WriteLine("Error: Selected shipping option is not available for this package.");
        }
    }
}

public class Program
{
    public static void Main()
    {
        Order order1 = new Order(2001, 3.0, "Ivan Petrov", 8000m, true);
        Order order2 = new Order(2002, 1.2, "Elena Sidorova", 2000m, false);

        // Создаем опции доставки
        ExpressDelivery express = new ExpressDelivery(101, "VIP Express", 600m, 15.0, 6, 300m);
        express.AssignCourier("Dmitry", "+7-999-111-2233");

        // Работа с явной реализацией интерфейса ITrackable через приведение типа
        ITrackable expressTrackable = express;
        expressTrackable.TrackingNumber = "EXP-777-XYZ";

        Pickup pickup = new Pickup(201, "Central Point", 150m, 30.0, "Nevsky Prospect, 10");
        ITrackable pickupTrackable = pickup;
        pickupTrackable.TrackingNumber = "PKP-555-ABC";

        // Внедрение зависимостей (Dependency Injection)
        OrderProcessingService expressService = new OrderProcessingService(express);
        expressService.ProcessOrder(order1);

        Console.WriteLine("\n=================================");

        OrderProcessingService pickupService = new OrderProcessingService(pickup);
        pickupService.ProcessOrder(order2);
    }
}